---
title: "数论——大数处理"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

In [1]:
#| code-fold: true
import math

## 📝 题目 50：快速幂算法 Pow(x, n)

::: {.callout-caution}
### 📖 题目描述
实现 `pow(x, n)` ，即计算 `x` 的整数 `n` 次幂函数（即，$x^n$）。

**输入输出示例**：

* **示例 1**：
    * **输入**：`x = 2.00000, n = 10`
    * **输出**：`1024.00000`

* **示例 2**：
    * **输入**：`x = 2.10000, n = 3`
    * **输出**：`9.26100`

* **示例 3**：
    * **输入**：`x = 2.00000, n = -2`
    * **输出**：`0.25000`
    * **解释**：$2^{-2} = (1/2)^2 = 1/4 = 0.25$

**⚠️ 数据范围警告**：
`-2^31 <= n <= 2^31 - 1` （这意味着 `n` 可以高达 21 亿！如果你用 `for` 循环乘 21 亿次，必定 Time Limit Exceeded 超时报错！）
:::

In [5]:
class Solution50:
    def myPow(self, x: float, n: int) -> float:
        res = 1.00
        if n < 0:  # 处理负数指数
            x = 1/ x
            n *= -1
        while n > 0:
            if n & 1:  # 如果 n 的当前末尾是 1，就把当前阶段的 x 乘进结果里
                res *= x
            x *= x  # x 自己要升级翻倍
            n >>= 1  # n 右移一位，看下一位
        return res

In [6]:
#| code-fold: true
def test_50(func):
    test_cases = [
        {"desc": "示例 1: 常规正数幂", "x": 2.00000, "n": 10, "expected": 1024.00000},
        {"desc": "示例 2: 小数底数", "x": 2.10000, "n": 3, "expected": 9.26100},
        {"desc": "示例 3: 负数幂 (变倒数)", "x": 2.00000, "n": -2, "expected": 0.25000},
        {"desc": "边界: 任何数的 0 次幂都是 1", "x": 99.99, "n": 0, "expected": 1.00000},
        {"desc": "边界: 底数为 1 的极限次幂", "x": 1.00000, "n": 2147483647, "expected": 1.00000}, # 如果用 for 循环会直接超时
        {"desc": "边界: 底数为 -1，奇数次", "x": -1.00000, "n": 999999, "expected": -1.00000},
        {"desc": "边界: 底数为 -1，偶数次", "x": -1.00000, "n": 1000000, "expected": 1.00000},
        {"desc": "极限: INT_MIN (Java/C++ 的梦魇)", "x": 2.00000, "n": -2147483648, "expected": 0.00000}, # 2 的极其负的次方，接近于 0
        {"desc": "负底数，偶数次幂", "x": -2.00000, "n": 4, "expected": 16.00000},
        {"desc": "负底数，奇数次幂", "x": -3.00000, "n": 3, "expected": -27.00000}
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期 (约)':<12} | {'实际 (约)':<12} | {'描述'}")
    print("-" * 75)

    for i, tc in enumerate(test_cases):
        actual = func(tc["x"], tc["n"])
        expected = tc["expected"]
        is_pass = math.isclose(actual, expected, rel_tol=1e-5, abs_tol=1e-5)

        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<3} | {status:<6} | {expected:<12.5f} | {actual:<12.5f} | {tc['desc']}")

    print("-" * 75)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_50(Solution50().myPow)

ID  | 结果     | 预期 (约)       | 实际 (约)       | 描述
---------------------------------------------------------------------------
1   | ✅ PASS | 1024.00000   | 1024.00000   | 示例 1: 常规正数幂
2   | ✅ PASS | 9.26100      | 9.26100      | 示例 2: 小数底数
3   | ✅ PASS | 0.25000      | 0.25000      | 示例 3: 负数幂 (变倒数)
4   | ✅ PASS | 1.00000      | 1.00000      | 边界: 任何数的 0 次幂都是 1
5   | ✅ PASS | 1.00000      | 1.00000      | 边界: 底数为 1 的极限次幂
6   | ✅ PASS | -1.00000     | -1.00000     | 边界: 底数为 -1，奇数次
7   | ✅ PASS | 1.00000      | 1.00000      | 边界: 底数为 -1，偶数次
8   | ✅ PASS | 0.00000      | 0.00000      | 极限: INT_MIN (Java/C++ 的梦魇)
9   | ✅ PASS | 16.00000     | 16.00000     | 负底数，偶数次幂
10  | ✅ PASS | -27.00000    | -27.00000    | 负底数，奇数次幂
---------------------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

如果你要算 $x^{13}$，暴力解法是 $x \cdot x \cdot x \dots$ 乘 13 次，复杂度 $O(N)$。
但**快速幂 (Fast Exponentiation)** 的思想是：我们把指数 $13$ 拆成**二进制**！
13 的二进制是 `1101`，也就是说 $13 = 8 + 4 + 1$。
所以：$x^{13} = x^8 \cdot x^4 \cdot x^1$。

**我们只需要这么做：**
1. **预处理负数**：如果 $n < 0$，先把 $x$ 变成 $1/x$，把 $n$ 变成 $-n$。（注意 Python 里取绝对值直接 `n = abs(n)` 即可）。
2. **初始化**：定一个结果变量 `res = 1.0`。
3. **核心循环（把 n 当作二进制不断右移）**：
   当 `n > 0` 时：
   - 看看 $n$ 的最后一位是不是 1（用 `n % 2 == 1` 或位运算 `n & 1`）。如果是，说明当前的 $x$ 需要被乘进结果里：`res *= x`。
   - 然后，让 $x$ 自己平方升级！`x *= x` （比如从 $x^1$ 升级成 $x^2$，再升级成 $x^4$，$x^8$）。
   - 最后，把 $n$ 砍掉一半，相当于右移一位：`n //= 2` （或者 `n >>= 1`）。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(\log n)$。不管 $n$ 是 21 亿还是多少，每次循环 $n$ 都砍半，最多只需要循环 31 次就能算出结果！这是真正意义上的降维打击。
* **空间复杂度**：$O(1)$。只需要常数级别的变量存放 `res` 和 `x`。
:::

## 📝 题目 1922：统计好数字的数目

::: {.callout-caution}
### 📖 题目描述
我们定义一个数字字符串是 **好数字**，当且仅当它的：
- **偶数下标**处的数字为 **偶数**（`0`, `2`, `4`, `6`, `8`，共 5 个选择）。
- **奇数下标**处的数字为 **质数**（`2`, `3`, `5`, `7`，共 4 个选择）。
*(注意：下标从 0 开始。)*

给你一个整数 `n` ，请你返回长度为 `n` 的好数字字符串的总数。
由于答案可能会很大，请将结果对 **`10^9 + 7`** 取余后返回。

**输入输出示例**：

* **示例 1**：
    * **输入**：`n = 1`
    * **输出**：`5`
    * **解释**：长度为 1，只有下标 0（偶数下标）。有 5 个选择。

* **示例 2**：
    * **输入**：`n = 4`
    * **输出**：`400`
    * **解释**：下标 0, 1, 2, 3。偶数下标有 2 个，奇数下标有 2 个。总数 = $5^2 \times 4^2 = 25 \times 16 = 400$。

* **示例 3**：
    * **输入**：`n = 50`
    * **输出**：`564908303`

**⚠️ 极限数据警告**：
`1 <= n <= 10^15` （$n$ 高达 1000 万亿！必须用快速幂！）
:::

In [7]:
class Solution1922:
    def countGoodNumbers(self, n: int) -> int:
        mod = 10 ** 9 + 7
        return pow(5, (n + 1) >> 1, mod) * pow(4, n >> 1, mod) % mod

In [9]:
#| code-fold: true
def test_1922(func):
    test_cases = [
        {"desc": "示例 1: 只有 1 位", "n": 1, "expected": 5},
        {"desc": "示例 2: 常规偶数位", "n": 4, "expected": 400},
        {"desc": "示例 3: 稍大数字", "n": 50, "expected": 564908303},
        {"desc": "边界: 最短长度", "n": 2, "expected": 20}, # 5 * 4 = 20
        {"desc": "极限 1: 十万级别", "n": 100000, "expected": 86331955},
        {"desc": "极限 2: 满级 Boss (1000万亿)", "n": 1000000000000000, "expected": 711414395} # 哪怕是 10^15，用快速幂也瞬间出结果
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期':<10} | {'实际':<10} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(tc["n"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<3} | {status:<6} | {tc['expected']:<10} | {actual:<10} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_1922(Solution1922().countGoodNumbers)

ID  | 结果     | 预期         | 实际         | 描述
-----------------------------------------------------------------
1   | ✅ PASS | 5          | 5          | 示例 1: 只有 1 位
2   | ✅ PASS | 400        | 400        | 示例 2: 常规偶数位
3   | ✅ PASS | 564908303  | 564908303  | 示例 3: 稍大数字
4   | ✅ PASS | 20         | 20         | 边界: 最短长度
5   | ✅ PASS | 86331955   | 86331955   | 极限 1: 十万级别
6   | ✅ PASS | 711414395  | 711414395  | 极限 2: 满级 Boss (1000万亿)
-----------------------------------------------------------------
测试总结: 通过 6/6


::: {.callout-important}
### 💡 思路讲解

这道题是典型的**“组合数学乘法原理” + “数论快速幂”**的二连击。

**第一步：组合数数（找规律）**
长度为 `n` 的字符串里：
- 有多少个偶数下标？答案是：`(n + 1) // 2` 个。每个位置有 5 种选择。
- 有多少个奇数下标？答案是：`n // 2` 个。每个位置有 4 种选择。
- 根据乘法原理，总方案数就是：$5^{\text{偶数下标个数}} \times 4^{\text{奇数下标个数}}$。

**第二步：数论降维（带取模的快速幂）**
题目要求对 $MOD = 10^9 + 7$ 取模。
你只需要把你刚刚写好的 `myPow` 函数稍微改造一下，加一个 `mod` 参数：`def myPow(x, n, mod):`
在循环里，每一次变大之后，立刻用镰刀割掉：
- `res = (res * x) % mod`
- `x = (x * x) % mod`

最后，把算出来的 $5^{\text{偶数个数}}$ 和 $4^{\text{奇数个数}}$ 再乘起来，**最后再取一次模**！
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(\log n)$。虽然 $n$ 是 $10^{15}$，但在大数快速幂面前，只需循环大约 50 次就能出结果！
* **空间复杂度**：$O(1)$。只用了常数级别的额外变量。
:::